# 1. Интервалы из ошибок модели: аналитические, эмпирические, бутстреп

**Сквозной вопрос блока:** модель говорит 95% — а сколько на самом деле?

В этом ноутбуке:
- кратко повторяем метрики интервалов (coverage, width, Winkler, pinball);
- смотрим на аналитические интервалы statsforecast — почему покрытие ниже номинала;
- строим интервалы из ошибок на кросс-валидации (ETNA): одна σ vs σ по горизонту vs эмпирические квантили;
- бутстреп остатков и бэггинг рядов (sktime);
- итог: остатки in-sample слишком оптимистичны → нужна калибровка → переход к конформным методам.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA

from etna.datasets import TSDataset
from etna.models import CatBoostMultiSegmentModel
from etna.pipeline import Pipeline
from etna.transforms import LagTransform, DateFlagsTransform
from etna.analysis import prediction_actual_scatter_plot
from etna.analysis.forecast import metric_per_segment_distribution_plot

from sktime.forecasting.compose import BaggingForecaster
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.arima import AutoARIMA as sktimeAutoARIMA
from sktime.transformations.bootstrap import MovingBlockBootstrapTransformer, STLBootstrapTransformer

from utilsforecast.losses import (
    coverage as coverage_metric,
    scaled_crps,
    winkler_score,
    mqloss
)

sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 4)

## 0. Данные и краткое повторение метрик интервалов

Используем часовые данные потребления электроэнергии AEP, агрегированные до дневного уровня.
Один ряд на весь блок — сотни скользящих окон для надёжной оценки покрытия.

In [ ]:
# Загрузка и агрегация до дня
df_hour = pd.read_csv('../data/AEP_hourly.csv', parse_dates=['Datetime'])
df_hour.columns = ['timestamp', 'target']

df_day = df_hour.resample('D', on='timestamp').sum().reset_index()
df_day = df_day.rename(columns={'timestamp': 'ds', 'target': 'y'})

print(f"Дневных наблюдений: {len(df_day)}, период {df_day['ds'].min().date()} — {df_day['ds'].max().date()}")

plt.figure(figsize=(14, 4))
plt.plot(df_day['ds'], df_day['y'], linewidth=0.8)
plt.title('AEP: дневное потребление')
plt.ylabel('МВт·ч')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Метрики интервалов (краткое повторение)

- **Coverage** (эмпирическое покрытие): доля точек внутри интервала. Для номинального уровня α=0.95 ожидаем ≈95%.
- **Width** (средняя ширина): чем уже интервал при том же покрытии, тем лучше.
- **Winkler score**: штраф за ширину + дополнительный штраф за промахи. Чем меньше, тем лучше.
- **Pinball loss** (для квантилей τ): ассиметричная функция потерь, штрафует недооценку сильнее при τ>0.5.

**Калибровка vs резкость**: калибровка = покрытие близко к номиналу; резкость = интервал узкий.

**Почему одно окно ничего не доказывает**: на H=14 и coverage=0.93 разница в 1 точку — это 7%. Нужно много окон.

In [ ]:
def evaluate_intervals_rolling(y_true, y_pred, lower, upper, level=0.95, n_windows=50):
    """
    Оценка интервалов на скользящих окнах.

    y_true, y_pred, lower, upper: pd.Series с индексом datetime.
    Возвращает словарь с метриками, усреднёнными по окнам.
    """
    H = len(y_pred) // n_windows
    if H < 7:
        H = 7

    coverages, widths, winklers, pinballs_lower, pinballs_upper = [], [], [], [], []

    for i in range(n_windows):
        start = i * H
        end = start + H
        if end > len(y_pred):
            break

        yt = y_true.iloc[start:end].values
        yp = y_pred.iloc[start:end].values
        lo = lower.iloc[start:end].values
        up = upper.iloc[start:end].values

        # Coverage
        inside = ((yt >= lo) & (yt <= up)).mean()
        coverages.append(inside)

        # Width
        widths.append((up - lo).mean())

        # Winkler
        alpha = 1 - level
        winkler = (up - lo) + (2/alpha) * (lo - yt) * (yt < lo) + (2/alpha) * (yt - up) * (yt > up)
        winklers.append(winkler.mean())

        # Pinball для квантилей
        tau_lower = (1 - level) / 2
        tau_upper = (1 + level) / 2
        pinball_l = np.where(yt >= lo, tau_lower * (yt - lo), (1 - tau_lower) * (lo - yt)).mean()
        pinball_u = np.where(yt >= up, tau_upper * (yt - up), (1 - tau_upper) * (up - yt)).mean()
        pinballs_lower.append(pinball_l)
        pinballs_upper.append(pinball_u)

    return {
        'coverage': np.mean(coverages),
        'coverage_std': np.std(coverages),
        'width': np.mean(widths),
        'winkler': np.mean(winklers),
        'n_windows': len(coverages)
    }

print("Хелпер evaluate_intervals_rolling готов — будет использоваться для всех методов блока.")

## 1. Отправная точка: аналитические интервалы statsforecast

AutoETS и AutoARIMA возвращают аналитические интервалы, основанные на предположениях:
- модель корректна;
- ошибки нормальны и гомоскедастичны;
- параметры известны точно.

На практике покрытие часто ниже номинала.

In [ ]:
# Разбиение: train первые 80%, test остаток
split_point = int(len(df_day) * 0.8)
train = df_day.iloc[:split_point].copy()
test = df_day.iloc[split_point:].copy()

train['unique_id'] = 'AEP'
test['unique_id'] = 'AEP'

print(f"Train: {len(train)} дней, test: {len(test)} дней")

In [ ]:
# Прогноз AutoETS с интервалами
sf = StatsForecast(
    models=[AutoETS(season_length=7)],
    freq='D',
    n_jobs=1
)

sf.fit(train[['unique_id', 'ds', 'y']])
forecast = sf.predict(h=len(test), level=[95])

# Извлечение интервалов
y_true_test = test['y'].values
y_pred = forecast['AutoETS'].values
lower = forecast['AutoETS-lo-95'].values
upper = forecast['AutoETS-hi-95'].values

# Визуализация на первых 60 днях
plt.figure(figsize=(14, 4))
plt.plot(test['ds'].iloc[:60], y_true_test[:60], 'o-', label='Факт', markersize=3)
plt.plot(test['ds'].iloc[:60], y_pred[:60], 's-', label='Прогноз', markersize=3)
plt.fill_between(test['ds'].iloc[:60], lower[:60], upper[:60], alpha=0.25, label='95% интервал')
plt.legend()
plt.title('AutoETS: аналитические интервалы (первые 60 дней теста)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Оценка на скользящих окнах
metrics_ets = evaluate_intervals_rolling(
    test['y'].reset_index(drop=True),
    pd.Series(y_pred),
    pd.Series(lower),
    pd.Series(upper),
    level=0.95,
    n_windows=50
)

print("AutoETS аналитические интервалы:")
print(f"  Coverage: {metrics_ets['coverage']:.3f} ± {metrics_ets['coverage_std']:.3f} (номинал 0.950)")
print(f"  Width: {metrics_ets['width']:.1f}")
print(f"  Winkler: {metrics_ets['winkler']:.1f}")
print(f"  Окон: {metrics_ets['n_windows']}")

**Причины занижения покрытия:**
- неопределённость параметров (σ, тренд, сезонность) не учтена;
- модель может быть неверной (ETS не идеально описывает ряд);
- ошибки не строго нормальны или гетероскедастичны.

## 2. Карта блока: таблица допущений

Методы интервалов различаются по трём осям:
- **форма распределения ошибок**: нормальное / произвольное;
- **корректность модели**: параметрическая модель верна / не обязательно;
- **перестановочность**: ошибки i.i.d. / могут быть зависимыми или нестационарными.

Таблица заполняется по ходу блока:

In [ ]:
assumptions_table = pd.DataFrame({
    'Метод': [
        'Аналитические (ETS/ARIMA)',
        'NaiveVariance (ETNA)',
        'EmpiricalPredictionIntervals',
        'Бутстреп остатков',
        'Бэггинг (block bootstrap)',
        'Конформные (Split CP)',
        'Квантильная регрессия'
    ],
    'Форма распределения': [
        'Нормальное',
        'Нормальное',
        'Произвольное',
        'Произвольное',
        'Произвольное',
        'Произвольное',
        'Произвольное'
    ],
    'Корректность модели': [
        'Да',
        'Не обязательно',
        'Не обязательно',
        'Не обязательно',
        'Не обязательно',
        'Не обязательно',
        'Не обязательно'
    ],
    'Перестановочность': [
        'Да (i.i.d.)',
        'Нет (зависимые)',
        'Нет',
        'Да (остатки i.i.d.)',
        'Нет (блоки)',
        'Да (калибровка)',
        'Нет'
    ]
})

print(assumptions_table.to_string(index=False))

## 3. Интервалы из ошибок на кросс-валидации

Вместо аналитических формул используем ошибки на кросс-валидации:
- получаем остатки на отложенных данных;
- считаем σ или квантили по каждому горизонту отдельно;
- применяем к будущему прогнозу.

**Основная идея:** σ(h) растёт с горизонтом, а одна общая σ этого не учитывает.

In [ ]:
# Упрощённая демонстрация: руками считаем интервалы из остатков кросс-валидации
from statsforecast.core import StatsForecast
from statsforecast.models import Naive

# Используем простую модель Naive для демонстрации
horizon_cv = 14
n_folds = 20

# Скользящая валидация
residuals_by_horizon = {h: [] for h in range(1, horizon_cv + 1)}

for fold in range(n_folds):
    fold_end = split_point - (n_folds - fold) * horizon_cv
    if fold_end < 100:
        continue

    train_fold = df_day.iloc[:fold_end].copy()
    test_fold = df_day.iloc[fold_end:fold_end + horizon_cv].copy()

    train_fold['unique_id'] = 'AEP'
    test_fold['unique_id'] = 'AEP'

    # Прогноз
    sf_fold = StatsForecast(models=[Naive()], freq='D', n_jobs=1)
    sf_fold.fit(train_fold[['unique_id', 'ds', 'y']])
    fc_fold = sf_fold.predict(h=horizon_cv, level=None)

    # Остатки по горизонтам
    for h in range(1, min(horizon_cv + 1, len(test_fold) + 1)):
        residual = test_fold['y'].iloc[h - 1] - fc_fold['Naive'].iloc[h - 1]
        residuals_by_horizon[h].append(residual)

In [ ]:
# Вычисляем σ и квантили по каждому горизонту
sigma_by_h = [np.std(residuals_by_horizon[h]) for h in range(1, horizon_cv + 1)]
q025_by_h = [np.quantile(residuals_by_horizon[h], 0.025) for h in range(1, horizon_cv + 1)]
q975_by_h = [np.quantile(residuals_by_horizon[h], 0.975) for h in range(1, horizon_cv + 1)]

In [ ]:
# Визуализация: σ растёт с горизонтом
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(range(1, horizon_cv + 1), sigma_by_h, 'o-')
axes[0].set_xlabel('Горизонт')
axes[0].set_ylabel('σ остатков')
axes[0].set_title('Стандартное отклонение ошибок по горизонту')
axes[0].grid(alpha=0.3)

axes[1].fill_between(range(1, horizon_cv + 1), q025_by_h, q975_by_h, alpha=0.3)
axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_xlabel('Горизонт')
axes[1].set_ylabel('Квантили остатков')
axes[1].set_title('95% интервал остатков по горизонту')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"σ на h=1: {sigma_by_h[0]:.1f}, на h={horizon_cv}: {sigma_by_h[-1]:.1f} — рост {sigma_by_h[-1]/sigma_by_h[0]:.1f}×")

In [ ]:
# Применяем интервалы к тестовому прогнозу
sf_final = StatsForecast(models=[AutoETS(season_length=7)], freq='D', n_jobs=1)
sf_final.fit(train[['unique_id', 'ds', 'y']])
fc_final = sf_final.predict(h=horizon_cv, level=None)

# Интервалы: прогноз ± квантили остатков
lower_cv = fc_final['AutoETS'].values + q025_by_h
upper_cv = fc_final['AutoETS'].values + q975_by_h

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(range(horizon_cv), test['y'].values[:horizon_cv], 'o-', label='Факт', markersize=4)
plt.plot(range(horizon_cv), fc_final['AutoETS'].values, 's-', label='AutoETS прогноз', markersize=4)
plt.fill_between(range(horizon_cv), lower_cv, upper_cv, alpha=0.3, label='95% (квантили CV)')
plt.legend()
plt.title('Интервалы из эмпирических квантилей остатков кросс-валидации')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Интервал расширяется с горизонтом и может быть асимметричным.")

## 4. Бутстреп — два источника неопределённости

### 4.1. Бутстреп остатков (руками)

Симулируем траектории будущего: берём модель, прогнозируем на 1 шаг, добавляем случайный остаток из истории, повторяем.
Это учитывает **неопределённость шума**, но не неопределённость параметров.

In [ ]:
# Простая демонстрация: ETS на statsforecast, бутстреп остатков
from statsforecast.models import ETS

# Обучаем модель
model_ets = ETS(season_length=7, model='ZZZ')
sf_boot = StatsForecast(models=[model_ets], freq='D', n_jobs=1)
sf_boot.fit(train[['unique_id', 'ds', 'y']])

# Остатки in-sample
fitted_values = sf_boot.fitted_[0, 0].fitted
residuals = train['y'].values - fitted_values
residuals = residuals[~np.isnan(residuals)]

print(f"Остатков для бутстрепа: {len(residuals)}")

In [ ]:
# Симуляция траекторий
n_simulations = 200
horizon_boot = 14
simulations = []

for _ in range(n_simulations):
    # Прогноз без интервалов
    fc = sf_boot.predict(h=horizon_boot, level=None)
    point_forecast = fc['ETS'].values

    # Добавляем случайные остатки
    bootstrapped_errors = np.random.choice(residuals, size=horizon_boot, replace=True)
    trajectory = point_forecast + bootstrapped_errors
    simulations.append(trajectory)

simulations = np.array(simulations)

# Квантили
lower_boot = np.quantile(simulations, 0.025, axis=0)
upper_boot = np.quantile(simulations, 0.975, axis=0)

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(range(horizon_boot), y_true_test[:horizon_boot], 'o-', label='Факт', markersize=4)
plt.plot(range(horizon_boot), simulations[:50].T, alpha=0.05, color='gray')
plt.plot(range(horizon_boot), point_forecast, 's-', label='Прогноз', markersize=4, color='red')
plt.fill_between(range(horizon_boot), lower_boot, upper_boot, alpha=0.3, label='95% бутстреп остатков')
plt.legend()
plt.title('Бутстреп остатков: 200 траекторий')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2. Бэггинг рядов (sktime BaggingForecaster)

Вместо бутстрепа остатков делаем бутстреп **самого ряда**: создаём синтетические версии исходного ряда, обучаем на них модели, усредняем прогнозы.
Это учитывает **неопределённость параметров/модели**.

In [ ]:
# Подготовка данных для sktime
y_sktime = train.set_index('ds')['y']
y_test_sktime = test.set_index('ds')['y']

fh = ForecastingHorizon(np.arange(1, horizon_boot + 1))

In [ ]:
# BaggingForecaster с MovingBlockBootstrapTransformer
from sktime.forecasting.arima import ARIMA

base_forecaster = ARIMA(order=(2, 1, 2), seasonal_order=(1, 0, 1, 7), suppress_warnings=True)

bootstrap_transformer = MovingBlockBootstrapTransformer(block_length=7)

bagging_forecaster = BaggingForecaster(
    forecaster=base_forecaster,
    bootstrap_transformer=bootstrap_transformer,
    n_estimators=100
)

bagging_forecaster.fit(y_sktime)
y_pred_bagging = bagging_forecaster.predict(fh)
pred_intervals_bagging = bagging_forecaster.predict_quantiles(fh=fh, alpha=[0.025, 0.975])

lower_bagging = pred_intervals_bagging.iloc[:, 0].values
upper_bagging = pred_intervals_bagging.iloc[:, 1].values

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(range(horizon_boot), y_test_sktime.values[:horizon_boot], 'o-', label='Факт')
plt.plot(range(horizon_boot), y_pred_bagging.values, 's-', label='Bagging прогноз')
plt.fill_between(range(horizon_boot), lower_bagging, upper_bagging, alpha=0.3, label='95% Bagging')
plt.legend()
plt.title('sktime BaggingForecaster + MovingBlockBootstrap')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3. Зачем блоки: короткая демонстрация на AR(1)

Если ряд имеет автокорреляцию, i.i.d.-бутстреп занижает разброс. Блочный бутстреп сохраняет локальную структуру.

In [ ]:
# Генерация AR(1)
np.random.seed(42)
n = 300
phi = 0.7
ar1_series = [0]

for _ in range(n - 1):
    ar1_series.append(phi * ar1_series[-1] + np.random.normal(0, 1))

ar1_series = np.array(ar1_series)

In [ ]:
# Наивная модель: последнее значение
last_value = ar1_series[-1]

# i.i.d. бутстреп остатков
residuals_ar1 = np.diff(ar1_series) - phi * ar1_series[:-1]
iid_bootstrap_samples = last_value + np.random.choice(residuals_ar1, size=(1000, 10), replace=True)
iid_std = iid_bootstrap_samples.std(axis=0)

# Блочный бутстреп
block_samples = []
for _ in range(1000):
    start = np.random.randint(0, len(ar1_series) - 10)
    block_samples.append(ar1_series[start:start+10] - ar1_series[start] + last_value)

block_samples = np.array(block_samples)
block_std = block_samples.std(axis=0)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(iid_std, 'o-', label='i.i.d. бутстреп')
plt.plot(block_std, 's-', label='Блочный бутстреп')
plt.xlabel('Горизонт')
plt.ylabel('Стандартное отклонение')
plt.title('AR(1): i.i.d. vs блочный бутстреп')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Блочный бутстреп даёт больший разброс — корректно учитывает автокорреляцию.")

## 5. Вывод и исправление таблицы API недели 2

**Основной вывод блока:**
- остатки in-sample слишком оптимистичны (модель видела эти данные);
- даже остатки на кросс-валидации могут быть смещены;
- **нужна калибровка на отложенных данных** → конформные интервалы (следующий ноутбук).

**Исправление таблицы API из недели 2:**
- statsforecast `prediction_intervals=ConformalIntervals(...)` — это конформные интервалы, а не бутстреп;
- ETNA `EmpiricalPredictionIntervals` — эмпирические квантили остатков, а не бутстреп;
- бутстреп есть в sktime (`BaggingForecaster`).

In [ ]:
# Финальная таблица методов блока
methods_summary = pd.DataFrame({
    'Метод': [
        'Аналитические',
        'NaiveVariance',
        'Empirical',
        'Бутстреп остатков',
        'Bagging (block bootstrap)'
    ],
    'Библиотека': [
        'statsforecast',
        'ETNA',
        'ETNA',
        'руками',
        'sktime'
    ],
    'Допущения': [
        'Нормальность, модель верна',
        'Нормальность, σ по h',
        'Произвольное распределение',
        'i.i.d. остатки',
        'Автокорреляция, неопр. параметров'
    ],
    'Покрытие (пример)': [
        f"{metrics_ets['coverage']:.3f}",
        '—',
        '—',
        '—',
        '—'
    ]
})

print("\nИтоговая таблица методов:")
print(methods_summary.to_string(index=False))

**Следующий шаг:** конформные интервалы — калибровка на отложенных данных, гарантии покрытия при перестановочности, адаптация к дрейфу (ACI/AgACI).